# 02 · Cleaning & QA

Runs QC checks on all raw parquets and writes cleaned files + a QC report.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

RAW = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
from cleaning import clean_dgstats

## 2.1 DGStats QC

In [ ]:
dgstats = pd.read_parquet(RAW / 'dgstats_raw.parquet')
print(f'Raw rows: {len(dgstats):,}')

# Duplicate check
dupes = dgstats.duplicated('application_id').sum()
print(f'Duplicate application_id rows: {dupes}')
dgstats = dgstats.drop_duplicates('application_id')

# Null critical fields
null_size = dgstats['system_size_dc'].isna().sum()
null_date = dgstats['app_approved_date'].isna().sum()
print(f'Null system_size_dc: {null_size} | Null app_approved_date: {null_date}')

# Assert <10% missing on key fields
for col in ['system_size_dc', 'app_approved_date', 'zip_code']:
    miss_pct = dgstats[col].isna().mean()
    assert miss_pct < 0.10, f'{col} missing rate {miss_pct:.1%} exceeds 10% threshold'
    print(f'{col}: {miss_pct:.2%} missing — OK')

In [ ]:
dgstats_clean = clean_dgstats(dgstats)
print(f'Clean rows: {len(dgstats_clean):,}')
print(f'Size outliers (>100 kW residential): {dgstats_clean["size_outlier"].sum()}')
dgstats_clean.to_parquet(PROC / 'dgstats_clean.parquet', index=False)

## 2.2 CAISO QC

In [ ]:
caiso = pd.read_parquet(RAW / 'caiso_hourly_raw.parquet')
caiso['interval_start_utc'] = pd.to_datetime(caiso['interval_start_utc'], utc=True)
caiso['year'] = caiso['interval_start_utc'].dt.year
caiso['hour'] = caiso['interval_start_utc'].dt.hour

# Row count per year — expect 8760 or 8784 (leap)
leap_years = {2016, 2020}
for yr, grp in caiso.groupby('year'):
    expected = 8784 if yr in leap_years else 8760
    actual = len(grp)
    status = 'OK' if actual == expected else f'WARN (expected {expected})'
    print(f'{yr}: {actual} rows — {status}')

In [ ]:
# Solar nighttime sanity check
night = caiso[caiso['hour'].isin(range(0, 6)) | caiso['hour'].isin(range(20, 24))]
nonzero_night_solar = (night['solar_mw'] > 0).sum()
if nonzero_night_solar > 0:
    print(f'WARNING: {nonzero_night_solar} nighttime rows with solar_mw > 0')
else:
    print('Nighttime solar sanity check: PASS')

# LMP range check
if 'lmp' in caiso.columns:
    extreme = caiso[(caiso['lmp'] < -500) | (caiso['lmp'] > 2000)]
    print(f'Extreme LMP rows (outside [-500, 2000]): {len(extreme)}')

caiso.to_parquet(PROC / 'caiso_clean.parquet', index=False)

## 2.3 QC Report

In [ ]:
qc_lines = [
    'QC Report — California Solar Grid Analysis',
    '=' * 50,
    f'DGStats clean rows: {len(dgstats_clean):,}',
    f'DGStats size outliers: {dgstats_clean["size_outlier"].sum()}',
    f'CAISO rows: {len(caiso):,}',
    f'Nighttime solar anomalies: {nonzero_night_solar}',
]

report_path = PROC / 'qc_report.txt'
report_path.write_text('\n'.join(qc_lines))
print('QC report written to', report_path)